In [1]:
import re
import camelot
import pandas as pd
import fitz
import pdfplumber
import statistics
from PyPDF2 import PdfReader


def parse_remittance(pdf_path, cliente):
    cliente = cliente.lower()

    # ==========================================================
    # ========================= CENCO ===========================
    # ==========================================================
    if cliente == "cenco":
        print(">> Cliente: CENCO")

        # 1) Intento Camelot stream agresivo
        try:
            tables_stream = camelot.read_pdf(
                pdf_path,
                pages="all",
                flavor="stream",
                row_tol=15,
                edge_tol=300,
                strip_text="\n",
                split_text=True,
                flag_size=True
            )
            df_all = pd.concat([t.df for t in tables_stream], ignore_index=True)
        except:
            df_all = pd.DataFrame()

        # Heurística para decidir fallback
        need_fallback = False
        if df_all.empty:
            need_fallback = True
        else:
            n_header_like = df_all.apply(
                lambda r: r.astype(str).str.upper().str.contains(
                    "DESCRIPCION|VALOR PAG|DOCUMENTO"
                ).any(), axis=1
            ).sum()
            if n_header_like / max(1, len(df_all)) > 0.6:
                need_fallback = True

        # 2) Fallback por texto
        if need_fallback:
            reader = PdfReader(pdf_path)
            pages_text = [p.extract_text() or "" for p in reader.pages]

            voucher_tokens = {
                "DAT","CH","DAV","DCA","DCC","DCF","DEV","DND","DPC",
                "FPM","FS","LTG","RPL","VOUCHER"
            }

            records = []
            for text in pages_text:
                lines = [ln.strip() for ln in text.splitlines() if ln.strip() != ""]
                buffer = []
                for ln in lines:
                    first_word = ln.split()[0] if ln.split() else ""
                    if first_word in voucher_tokens:
                        if buffer:
                            records.append(" ".join(buffer))
                        buffer = [ln]
                    else:
                        if buffer:
                            buffer.append(ln)
                        else:
                            buffer = [ln]
                if buffer:
                    records.append(" ".join(buffer))

            parsed = []
            date_re = r"\b\d{2}/\d{2}/\d{4}\b"
            doc_re = r"\b[Pp][A-Z0-9]*\d{4,}\b"
            num_re = r"-?\d[\d\.,]*"

            for rec in records:
                voucher = ""
                descripcion = ""
                documento = ""
                fecha = ""
                valor_pag = ""
                doc_soporte = ""

                parts = rec.split()
                voucher_tokens_list = list(voucher_tokens)

                if parts and parts[0] in voucher_tokens_list:
                    voucher = parts[0]
                    rest = " ".join(parts[1:])
                else:
                    m_v = re.search(r"\b(" + "|".join(voucher_tokens) + r")\b", rec)
                    voucher = m_v.group(1) if m_v else ""
                    rest = rec

                # Descripcion
                m_desc = re.search(
                    r"(FACTURA PROVEEDOR|FACTURA VENTA|COSTO DE TRANSFEREN|"
                    r"COSTO DE TRANSFERENCIA|DIF\. COSTO\/CANTIDAD|DESCRIPCION)",
                    rest, flags=re.IGNORECASE
                )
                if m_desc:
                    descripcion = m_desc.group(0).strip().upper()
                else:
                    descripcion = " ".join(rest.split()[:3]).upper()

                # Documento
                m_doc = re.search(doc_re, rec)
                documento = m_doc.group(0).strip().upper() if m_doc else ""

                # Fecha
                m_fecha = re.search(date_re, rec)
                fecha = m_fecha.group(0) if m_fecha else ""

                # Numeros
                valores = re.findall(num_re, rec)
                valores_clean = []
                for v in valores:
                    v2 = v.replace(".", "").replace(",", "")
                    if re.search(r"\d", v2):
                        valores_clean.append(v)

                valor_fac = iva_fac = None

                if len(valores_clean) >= 4:
                    doc_soporte = valores_clean[-1]
                    valor_pag = valores_clean[-2]
                elif len(valores_clean) == 3:
                    doc_soporte = valores_clean[-1]
                    valor_pag = valores_clean[-2]
                elif len(valores_clean) == 1:
                    valor_pag = valores_clean[0]

                parsed.append({
                    "VOUCHER": voucher,
                    "DESCRIPCION": descripcion,
                    "DOCUMENTO": documento,
                    "SECCION": "",
                    "F. REGISTRO": fecha,
                    "VALOR PAG": valor_pag,
                    "DOC.SOPORTE": doc_soporte
                })

            return pd.DataFrame(parsed)

        return df_all

    # ==========================================================
    # ========================= EURO ===========================
    # ==========================================================
    if cliente == "euro":
        print(">> Cliente: EURO")

        results = []

        with pdfplumber.open(pdf_path) as pdf:
            for pnum, page in enumerate(pdf.pages, start=1):

                words = page.extract_words()
                if not words:
                    continue

                # Detectar encabezados por posición (tu lógica tal cual)
                header_cands = []
                for w in words:
                    t = re.sub(r'\s+', ' ', w['text']).strip()
                    tl = t.lower().replace(' ', '')
                    if tl in ('reg','reg.') or 'detalle' in tl or 'doc' in tl or 'cruce' in tl \
                    or tl in ('c','c.','o','o.') or tl in (
                        'descuentos','retenciones','valorfactura','valorpago'
                    ):
                        header_cands.append({'text': t, 'x0': w['x0'], 'top': w['top']})

                if not header_cands:
                    continue

                header_cands.sort(key=lambda x: x['top'])

                clusters = []
                tol = 4
                for h in header_cands:
                    if not clusters or abs(h['top'] - clusters[-1]['top_mean']) > tol:
                        clusters.append({'items':[h], 'top_mean': h['top']})
                    else:
                        clusters[-1]['items'].append(h)
                        clusters[-1]['top_mean'] = statistics.mean(
                            [it['top'] for it in clusters[-1]['items']]
                        )

                cluster = max(clusters, key=lambda c: c['top_mean'])
                header_items = sorted(cluster['items'], key=lambda it: it['x0'])
                header_top = cluster['top_mean']

                positions = {}
                co_x_list, doc_x_list = [], []

                for it in header_items:
                    tnorm = it['text'].strip().lower().replace(' ','')
                    if 'reg' in tnorm:
                        positions['Reg.'] = it['x0']
                    elif 'detalle' in tnorm:
                        positions['Detalle'] = it['x0']
                    elif tnorm in ('c','c.','o','o.'):
                        co_x_list.append(it['x0'])
                    elif 'doc' in tnorm or 'cruce' in tnorm:
                        doc_x_list.append(it['x0'])
                    elif 'descuentos' in tnorm:
                        positions['Descuentos'] = it['x0']
                    elif 'retenciones' in tnorm:
                        positions['Retenciones'] = it['x0']
                    elif 'valorfactura' in tnorm:
                        positions['Valor Factura'] = it['x0']
                    elif 'valorpago' in tnorm:
                        positions['Valor Pago'] = it['x0']

                if not (positions.get('Reg.') and positions.get('Detalle') and doc_x_list):
                    continue

                positions['C.O.'] = statistics.mean(co_x_list) if co_x_list else (positions['Reg.'] + min(doc_x_list))/2
                positions['Doc.Cruce'] = statistics.mean(doc_x_list)

                centers_sorted = sorted(positions.items(), key=lambda x: x[1])
                colnames = [c[0] for c in centers_sorted]
                xs = [c[1] for c in centers_sorted]

                bounds = [0.0] + [(a+b)/2.0 for a,b in zip(xs, xs[1:])] + [page.width + 1.0]

                padding = 5
                for i, col in enumerate(colnames):
                    if col in ('Descuentos','Retenciones','Valor Factura','Valor Pago'):
                        bounds[i] -= padding
                        bounds[i+1] += padding

                data_words = [w for w in words if w['top'] > header_top - 2]
                data_words.sort(key=lambda w: (w['top'], w['x0']))

                rows = []
                current, last_top = [], None
                row_tol = 8
                for w in data_words:
                    if last_top is None or abs(w['top'] - last_top) <= row_tol:
                        current.append(w)
                        last_top = w['top'] if last_top is None else (last_top + w['top'])/2
                    else:
                        rows.append(current)
                        current = [w]
                        last_top = w['top']
                if current:
                    rows.append(current)

                parsed = []
                for row_words in rows:
                    cells = {name: [] for name in colnames}
                    for w in row_words:
                        x = w['x0']
                        col_idx = next((i for i in range(len(bounds)-1) if x >= bounds[i] and x < bounds[i+1]), None)
                        if col_idx is None:
                            col_idx = min(range(len(bounds)-1), key=lambda i: abs((bounds[i]+bounds[i+1])/2 - x))
                        cells[colnames[col_idx]].append(w['text'])

                    row_dict = {
                        'page': pnum,
                        'raw_top': statistics.mean([w['top'] for w in row_words])
                    }
                    for name in colnames:
                        row_dict[name] = ' '.join(cells[name]).strip()

                    parsed.append(row_dict)

                results.extend(parsed)

        return pd.DataFrame(results)

    # ==========================================================
    # =========================== D1 ===========================
    # ==========================================================
    if cliente == "d1":
        print(">> Cliente: D1")

        factura_pattern = re.compile(
            r"RE\s+\d+\s+PMP\d+\s+\d{1,3}(?:\.\d{3})*"
            r"\s+\d+\s+\d{1,3}(?:\.\d{3})*"
            r"\s+\d+\s+\d{1,3}(?:\.\d{3})*"
        )
        facturas = []

        with fitz.open(pdf_path) as doc:
            for page in doc:
                text = page.get_text()
                for match in factura_pattern.findall(text):
                    parts = match.split()
                    if len(parts) == 8:
                        facturas.append({
                            "TD": parts[0],
                            "Doc.Interno": parts[1],
                            "Referencia / Factura": parts[2],
                            "Valor Bruto": parts[3].replace('.', ''),
                            "Retenciones": parts[4],
                            "IVA": parts[5].replace('.', ''),
                            "Desc/Rec": parts[6],
                            "Neto Pagado": parts[7].replace('.', '')
                        })

        df = pd.DataFrame(facturas)
        df["Importe de Remittance"] = df["Neto Pagado"].astype(float)
        df["Tipo de Documento"] = "Factura"
        return df

    # Caso cliente inválido
    raise ValueError(f"Cliente no reconocido: {cliente}")


In [2]:
df_cenco = parse_remittance("Remittance_Cenco.pdf", "cenco")


>> Cliente: CENCO


FileNotFoundError: [Errno 2] No such file or directory: 'Remittance_Cenco.pdf'

In [1]:
# =====================================================
# 0. Importación de librerías y módulos utilitarios
# =====================================================

import os       # Manejo de rutas y directorios del sistema operativo
import sys      # Detección de ejecución empaquetada y manipulación de rutas del intérprete
import re       # Expresiones regulares (búsqueda y limpieza de texto)
import io       # Manejo de flujos de datos en memoria (buffers, streams)
import warnings # Control de advertencias del sistema y librerías externas

import numpy as np   # Operaciones numéricas y lógicas
import pandas as pd  # Manipulación y análisis de datos tabulares
import camelot       # Extracción de tablas desde PDFs
from PyPDF2 import PdfReader  # Lector PDF
from openpyxl import load_workbook  # Lector Excel (.xlsx)
#from utils import *  # Funciones utilitarias internas

# Configuración de advertencias
warnings.filterwarnings("ignore", category=UserWarning, module="camelot")


In [2]:
# =====================================================
# 1. Localización dinámica de la carpeta raíz del proyecto
# =====================================================
def _project_root():
    """
    Obtiene la ruta base del proyecto sin importar el entorno de ejecución.
    """
    if getattr(sys, "frozen", False):
        macos_dir = os.path.dirname(sys.executable)
        contents_dir = os.path.dirname(macos_dir)
        app_bundle = os.path.dirname(contents_dir)
        return os.path.dirname(app_bundle)
    return os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))


In [17]:
# 🔵 Ruta de la carpeta Remittance
ruta_remittance = "/Users/svonbergen/Desktop/Unilever/2.Automatizacion_aplicacion_pagos/Pruebas/Archivos/Remittance/Colombia"

# 🔵 Ruta de la carpeta FBL5N
ruta_fbl5n = "/Users/svonbergen/Desktop/Unilever/2.Automatizacion_aplicacion_pagos/Pruebas/Archivos/Cartera"

# ⛔️ MODIFICAR SOLO ESTAS DOS LÍNEAS:
archivo_remittance = os.path.join(ruta_remittance, "Remittance_Cenco.pdf")     # <-- poné tu nombre real
archivo_fbl5n      = os.path.join(ruta_fbl5n, "FBL5N.xlsx")                    # <-- poné tu nombre real

print("Usando estos archivos:")
print("REM:", archivo_remittance)
print("FBL5N:", archivo_fbl5n)

Usando estos archivos:
REM: /Users/svonbergen/Desktop/Unilever/2.Automatizacion_aplicacion_pagos/Pruebas/Archivos/Remittance/Colombia/Remittance_Cenco.pdf
FBL5N: /Users/svonbergen/Desktop/Unilever/2.Automatizacion_aplicacion_pagos/Pruebas/Archivos/Cartera/FBL5N.xlsx


In [3]:
# =====================================================
# 1. Lectura de Remittance
# =====================================================

archivo_remittance = "/Users/svonbergen/Desktop/Unilever/2.Automatizacion_aplicacion_pagos/Pruebas/Archivos/Remittance/Colombia/Remittance_Cenco.pdf"

reader = PdfReader(archivo_remittance)
num_pages = len(reader.pages)
print("Páginas del PDF:", num_pages)


Páginas del PDF: 39


In [4]:
# =====================================================
# 2. Extracción de tablas con Camelot
# =====================================================

tables_stream = camelot.read_pdf(
    archivo_remittance, pages="all", flavor="stream", strip_text="\n"
)

stream_pages = set(int(t.page) for t in tables_stream)
all_pages = set(range(1, num_pages + 1))
missing_pages = all_pages - stream_pages

tables_lattice = []
if missing_pages:
    tables_lattice = camelot.read_pdf(
        archivo_remittance,
        pages=",".join([str(p) for p in missing_pages]),
        flavor="lattice",
        strip_text="\n"
    )

df_all = pd.concat(
    [t.df for t in list(tables_stream) + list(tables_lattice)],
    ignore_index=True
)

print("Dimensión combinada:", df_all.shape)
df_all.head()


Dimensión combinada: (2293, 14)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,,,,,,AV 9 Nro 125-30 - TESORERIA,,,,,,,,NaN
1,,,,,,PAGINA: 1,,,,,,,,NaN
2,,,,,,FECHA : 12/11/2025,,,,,,,,NaN
3,VOUCHER,DESCRIPCION,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE,NaN
4,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0,NaN


In [5]:
# =====================================================
# 3. Detectar encabezado real (DOCUMENTO / VOUCHER)
# =====================================================

header_idx_candidates = df_all.apply(
    lambda row: row.astype(str).str.contains("DOCUMENTO", case=False).any() or
                row.astype(str).str.contains("VOUCHER", case=False).any(),
    axis=1
)

if not header_idx_candidates.any():
    raise ValueError("❌ ERROR: no se detectó encabezado en el PDF")

header_row_idx = header_idx_candidates.idxmax()
print("Encabezado detectado en fila:", header_row_idx)

df_all.columns = df_all.iloc[header_row_idx]
df_all = df_all.drop(index=list(range(0, header_row_idx + 1))).reset_index(drop=True)

df_all = df_all.loc[:, ~df_all.columns.isna()]
df_all.head()


Encabezado detectado en fila: 3


3,VOUCHER,DESCRIPCION,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
0,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0
1,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0
2,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0
3,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0
4,FS,FACTURA VENTA,VPP2 2022342ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0


In [7]:
    df_all["VALOR PAG"] = np.where(
        (df_all["VALOR PAG"] == 0) &
        (df_all["DOC.SOPORTE"].astype(str).str.strip() != ""),
        df_all["DOC.SOPORTE"],
        df_all["VALOR PAG"]
    )

In [8]:
df_all[df_all["DESCRIPCION"] == "FACTURA PROVEEDOR"].head(20)

3,VOUCHER,DESCRIPCION,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
1047,FPM,FACTURA PROVEEDOR,PMP1284464,PLAT - CROSSD,RANCHO,07/09/2025,-6.986.880,-1.106.256,0,0,0,0,-8.093.136
1049,FPM,FACTURA PROVEEDOR,PMP1284463,PLAT - CROSSD,RANCHO,07/09/2025,-3.732.265,-595.109,0,0,0,0,-4.327.374
1051,FPM,FACTURA PROVEEDOR,PMP1284461,PLAT - CROSSD,RANCHO,07/09/2025,-1.397.376,-221.251,0,0,0,0,-1.618.627
1053,FPM,FACTURA PROVEEDOR,PMP1282399,PLAT - CROSS,PERFUME,03/09/2025,-95.170.861,-18.082.463,0,0,0,0,-113.253.324
1055,FPM,FACTURA PROVEEDOR,PMP1282298,PLAT - CROSS,RANCHO,02/09/2025,-1.372.141,-217.256,0,0,0,0,-1.589.397
1057,FPM,FACTURA PROVEEDOR,PMP1282303,PLAT - CROSS,RANCHO,02/09/2025,-1.527.322,-241.826,0,0,0,0,-1.769.148
1095,FPM,FACTURA PROVEEDOR,PMP1284464,PLAT - CROSSD,RANCHO,07/09/2025,-6.986.880,-1.106.256,0,0,0,0,-8.093.136
1097,FPM,FACTURA PROVEEDOR,PMP1284463,PLAT - CROSSD,RANCHO,07/09/2025,-3.732.265,-595.109,0,0,0,0,-4.327.374
1099,FPM,FACTURA PROVEEDOR,PMP1284461,PLAT - CROSSD,RANCHO,07/09/2025,-1.397.376,-221.251,0,0,0,0,-1.618.627
1101,FPM,FACTURA PROVEEDOR,PMP1282399,PLAT - CROSS,PERFUME,03/09/2025,-95.170.861,-18.082.463,0,0,0,0,-113.253.324


In [18]:
# =====================================================
# 4. Normalizar columnas
# =====================================================

cols = pd.Series(df_all.columns.astype(str))
cols = cols.str.replace(r"[\n\r]+", " ", regex=True)
cols = cols.str.replace(r"\s+", " ", regex=True)
cols = cols.str.strip().str.upper()
df_all.columns = cols

print("Columnas normalizadas:")
print(df_all.columns.tolist())
df_all.head()


Columnas normalizadas:
['VOUCHER', 'DESCRIPCION', 'DOCUMENTOTIENDA', 'SECCION', 'F. REGISTRO', 'VALOR FAC.', 'IVA FAC.', 'RET. FUENTE', 'RET. IVA', 'RET. ICA', 'OTROS IMP.', 'VALOR PAG', 'DOC.SOPORTE']


3,VOUCHER,DESCRIPCION,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
0,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0
1,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0
2,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0
3,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0
4,FS,FACTURA VENTA,VPP2 2022342ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0


In [ ]:
# ----------------------------
# POST-PROCESADO ROBUSTO de df_all (pegar justo después de concatenar tablas)
# ----------------------------
import re

# Voucher tokens esperados (ajustá si hace falta)
voucher_tokens = {"DAT","CH","DAV","DCA","DCC","DCF","DEV","DND","DPC","FPM","FS","LTG","RPL","VOUCHER","DEC","VPP","RPC","DCR"}

# 1) Detectar header real dentro de df_all (fila que contiene VOUCHER o DESCRIPCION)
header_idx_candidates = df_all.apply(
    lambda row: row.astype(str).str.contains("VOUCHER", case=False).any()
             or row.astype(str).str.contains("DESCRIPCION", case=False).any()
             or row.astype(str).str.contains("DOCUMENTO", case=False).any(),
    axis=1
)

if not header_idx_candidates.any():
    print("⚠️ No se detectó header claro; seguimos con df_all tal cual.")
else:
    header_row_idx = header_idx_candidates.idxmax()
    # Aplicar header real
    df_all.columns = df_all.iloc[header_row_idx].astype(str)
    df_all = df_all.drop(index=list(range(0, header_row_idx + 1))).reset_index(drop=True)

# 2) Quitar columnas totalmente nulas
df_all = df_all.loc[:, ~df_all.columns.isna()]

# 3) Normalizar nombres de columnas: mayúsculas, eliminar saltos de línea y variantes comunes
cols = pd.Series(df_all.columns.astype(str))
cols = cols.str.replace(r"[\n\r]+", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip().str.upper()
cols = cols.str.replace("DOCUMENTO TIENDA", "DOCUMENTO", regex=False).str.replace("DOCUMENTOTIENDA", "DOCUMENTO", regex=False)
cols = cols.str.replace("VALOR PAG\\.", "VALOR PAG", regex=True).str.replace("VALOR PAGO", "VALOR PAG", regex=False)
df_all.columns = cols.values

# 4) Eliminar filas que sean exactamente el header repetido (ej: fila que tenga "VOUCHER" como primer campo)
first_col = df_all.columns[0]
mask_header_repeat = df_all[first_col].astype(str).str.upper().str.strip().isin(["VOUCHER", "VOUCHER "]) | \
                     df_all.apply(lambda r: r.astype(str).str.upper().str.contains("DESCRIPCION|VALOR PAG|DOCUMENTO").any(), axis=1)
# conservador: solo eliminar si la fila parece contener el header completo (todas columnas con valores de header)
# para no borrar filas válidas, comprobamos que la primera columna es 'VOUCHER' o que muchas columnas contienen palabras de header
df_all = df_all[~mask_header_repeat].reset_index(drop=True)

# 5) Reconstruir filas partidas:
# Regla: si la primera columna no contiene un voucher token conocido => se trata de continuación de la fila anterior
recs = []
current = None

def is_voucher_token(x):
    x = str(x).strip().upper()
    return any(x == vt for vt in voucher_tokens) or (x in voucher_tokens)

for idx, row in df_all.iterrows():
    first = str(row.iloc[0]).strip()
    if is_voucher_token(first):
        # inicio de nuevo registro
        if current is not None:
            recs.append(current)
        # copiamos la fila actual como base (convertir a dict para facilidad)
        current = row.astype(str).to_dict()
    else:
        # continuación: pegar columnas no vacías en current
        if current is None:
            # caso raro: no hay current, lo iniciamos con esta fila
            current = row.astype(str).to_dict()
        else:
            for col in df_all.columns:
                val_curr = str(current.get(col,"")).strip()
                val_row = str(row.get(col,"")).strip()
                # si la columna actual en current está vacía o es "nan", tomar de row
                if (val_curr == "" or val_curr.lower() in ["nan", "none"]) and val_row != "" and val_row.lower() not in ["nan","none"]:
                    current[col] = val_row
                # en columnas textuales (DOCUMENTO, SECCION) preferimos concatenar para no perder info
                elif col.upper() in ["DOCUMENTO","DOCUMENTOTIENDA","DESCRIPCION","SECCION"] and val_row != "":
                    # evitar duplicados al concatenar
                    if val_row not in val_curr:
                        if val_curr == "" or val_curr.lower() in ["nan","none"]:
                            current[col] = val_row
                        else:
                            current[col] = val_curr + " " + val_row

# añadir el último current
if current is not None:
    recs.append(current)

# crear nuevo df_all reconstruido
df_all = pd.DataFrame(recs, columns=df_all.columns)

# 6) Normalizar espacios y mayúsculas en columnas importantes
for c in df_all.columns:
    if df_all[c].dtype == object:
        df_all[c] = df_all[c].astype(str).str.replace(r"[\n\r]+", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()

# 7) Convertir VALOR PAG y DOC.SOPORTE a numérico robusto y reemplazar VALOR PAG==0 por DOC.SOPORTE si aplica
def parse_num_simple(x):
    if pd.isna(x): return np.nan
    s = str(x).strip()
    if s == "": return np.nan
    s = s.replace(" ", "").replace("(CR)","-").replace("(DR)","")
    if s.endswith("-"): s = "-" + s[:-1]
    s = re.sub(r"[^0-9\.,-]", "", s)
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except:
        return np.nan

for col in ["VALOR PAG", "DOC.SOPORTE"]:
    if col in df_all.columns:
        df_all[col + "_num"] = df_all[col].apply(parse_num_simple)
    else:
        df_all[col + "_num"] = np.nan

# Reemplazo: si VALOR PAG num es NaN or 0, pero DOC.SOPORTE_num tiene valor -> usarlo
mask_replace = (df_all["VALOR PAG_num"].fillna(0) == 0) & (df_all["DOC.SOPORTE_num"].notna())
if mask_replace.any():
    df_all.loc[mask_replace, "VALOR PAG_num"] = df_all.loc[mask_replace, "DOC.SOPORTE_num"]

# Sobrescribir las columnas originales por las numéricas limpias (opcional)
if "VALOR PAG_num" in df_all.columns:
    df_all["VALOR PAG"] = df_all["VALOR PAG_num"]
if "DOC.SOPORTE_num" in df_all.columns:
    df_all["DOC.SOPORTE"] = df_all["DOC.SOPORTE_num"]

# Eliminar columnas auxiliares num (si querés)
df_all = df_all.drop(columns=[c for c in df_all.columns if c.endswith("_num")], errors="ignore")

# 8) Filtrar sólo vouchers esperados en primer columna (opcional)
df_filtered = df_all[df_all[first_col].astype(str).str.upper().isin(voucher_tokens)].copy()
df_filtered = df_filtered.reset_index(drop=True)

print("Registros finales (reconstruidos):", df_filtered.shape[0])
display(df_filtered.head(40))

# Asigna df_all a df_filtered para que el flujo posterior use las filas reconstruidas
df_all = df_filtered


In [19]:
# =====================================================
# 5. Filtrar VOUCHERS válidos
# =====================================================

filter_values = [
    "DAT","CH","DAV","DCA","DCC","DCF","DEV","DND","DPC",
    "FPM","FS","LTG","RPL","VOUCHER"
]

remittance = df_all[df_all[df_all.columns[0]].isin(filter_values)].copy()
remittance = remittance.reset_index(drop=True)

print("Filtrado por VOUCHERS — filas:", remittance.shape[0])
remittance.head()


Filtrado por VOUCHERS — filas: 729


3,VOUCHER,DESCRIPCION,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE
0,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0
1,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0
2,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0
3,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0
4,FS,FACTURA VENTA,VPP2 2022342ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0


In [20]:
print(remittance.columns.tolist())


['VOUCHER', 'DESCRIPCION', 'DOCUMENTOTIENDA', 'SECCION', 'F. REGISTRO', 'VALOR FAC.', 'IVA FAC.', 'RET. FUENTE', 'RET. IVA', 'RET. ICA', 'OTROS IMP.', 'VALOR PAG', 'DOC.SOPORTE']


In [17]:
# =====================================================
# 6. Renombrar columnas clave
# =====================================================

renombres = {
    "DESCRIPCION": "Tipo de Documento",
    "DOCUMENTO": "Referencia / Factura",
    "VALOR PAG": "Importe de Remittance"
}

# Renombrar solo si existen
remittance = remittance.rename(columns={c: renombres[c] for c in renombres if c in remittance.columns})
print("Columnas tras renombrar:", remittance.columns.tolist())

# =====================================================
# 7. Seleccionar solo columnas finales
# =====================================================

columnas_finales = [
    "VOUCHER",
    "Tipo de Documento",
    "Referencia / Factura",
    "SECCION",
    "Importe de Remittance",
    "DOC.SOPORTE"
]

# Filtrar solo las columnas disponibles (evita KeyError si falta alguna)
columnas_existentes = [c for c in columnas_finales if c in remittance.columns]

remittance = remittance[columnas_existentes]

print("Columnas finales:", remittance.columns.tolist())
remittance.head(100)


Columnas tras renombrar: ['VOUCHER', 'Tipo de Documento', 'DOCUMENTOTIENDA', 'SECCION', 'F. REGISTRO', 'VALOR FAC.', 'IVA FAC.', 'RET. FUENTE', 'RET. IVA', 'RET. ICA', 'OTROS IMP.', 'Importe de Remittance', 'DOC.SOPORTE']
Columnas finales: ['VOUCHER', 'Tipo de Documento', 'SECCION', 'Importe de Remittance', 'DOC.SOPORTE']


3,VOUCHER,Tipo de Documento,SECCION,Importe de Remittance,DOC.SOPORTE
0,FS,FACTURA VENTA,,57.239.000,0
1,FS,FACTURA VENTA,,6.426.000,0
2,FS,FACTURA VENTA,,41.650.000,0
3,FS,FACTURA VENTA,,2.507.098.292,0
4,FS,FACTURA VENTA,,1.075.965.191,0


In [8]:
# =====================================================
# 7. Limpieza del importe — SOLO numérico
# =====================================================

def clean_importe(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor).strip()

    s = s.replace(" ", "").replace("(CR)", "-").replace("(DR)", "")
    if s.endswith("-"): s = "-" + s[:-1]

    s = re.sub(r"[^0-9,-]", "", s)
    s = s.replace(".", "")
    s = s.replace(",", ".")

    try:
        return float(s)
    except:
        return np.nan

remittance["Importe_clean"] = remittance["Importe de Remittance"].apply(clean_importe)

print("Importes nulos:", remittance["Importe_clean"].isna().sum())
remittance.head(15)


Importes nulos: 52


3,VOUCHER,Tipo de Documento,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,Importe de Remittance,DOC.SOPORTE,Importe_clean
0,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0,5.723900e+07
1,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0,6.426000e+06
2,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0,4.165000e+07
3,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0,2.507098e+09
4,FS,FACTURA VENTA,VPP2 2022342ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0,1.075965e+09
5,CH,Costo de Transferen,ADM. JUMBO - SEDE,,12/11/2025,3.000,0,0,0,0,0,3.000,0,3.000000e+03
6,VOUCHER,DESCRIPCION,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,VALOR PAG,DOC.SOPORTE,NaN
7,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0,5.723900e+07
8,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0,6.426000e+06
9,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0,4.165000e+07


In [9]:
# =====================================================
# 8. Depuración final del Remittance
# =====================================================

print("Filas originales en remittance:", remittance.shape[0])

# 1) Eliminar filas que son encabezados repetidos
mask_header_like = remittance.apply(
    lambda row: row.astype(str).str.contains("DESCRIPCION|VALOR PAG", case=False).any(),
    axis=1
)
remittance = remittance[~mask_header_like]

print("Después de quitar encabezados repetidos:", remittance.shape[0])

# 2) Quitar filas donde el importe es NaN
remittance = remittance[remittance["Importe_clean"].notna()]

print("Después de quitar importes NaN:", remittance.shape[0])

# 3) Quitar filas donde todo está vacío excepto tal vez un 0
mask_all_empty = remittance.apply(
    lambda r: all((str(x).strip() in ["", "0", "nan", "None"]) for x in r),
    axis=1
)
remittance = remittance[~mask_all_empty]

print("Después de quitar filas completamente vacías:", remittance.shape[0])

# 4) Reset de índice final
remittance = remittance.reset_index(drop=True)

print("\n=== Remittance limpio FINAL ===")
print(remittance.head(20))

print("\nTotal filas finales:", remittance.shape)


Filas originales en remittance: 729
Después de quitar encabezados repetidos: 677
Después de quitar importes NaN: 677
Después de quitar filas completamente vacías: 677

=== Remittance limpio FINAL ===
3  VOUCHER    Tipo de Documento                  DOCUMENTOTIENDA  \
0       FS        FACTURA VENTA  VPP1 2004529ADM. FIDELIDAD - SE   
1       FS        FACTURA VENTA  VPP1 2004621ADM. FIDELIDAD - SE   
2       FS        FACTURA VENTA  VPP1 2004622ADM. FIDELIDAD - SE   
3       FS        FACTURA VENTA    VPP2 2021716ADM. JUMBO - SEDE   
4       FS        FACTURA VENTA    VPP2 2022342ADM. JUMBO - SEDE   
5       CH  Costo de Transferen                ADM. JUMBO - SEDE   
6       FS        FACTURA VENTA  VPP1 2004529ADM. FIDELIDAD - SE   
7       FS        FACTURA VENTA  VPP1 2004621ADM. FIDELIDAD - SE   
8       FS        FACTURA VENTA  VPP1 2004622ADM. FIDELIDAD - SE   
9       FS        FACTURA VENTA    VPP2 2021716ADM. JUMBO - SEDE   
10      FS        FACTURA VENTA    VPP2 2022342ADM. 

In [10]:
# =====================================================
# 9. Limpieza final profunda del Remittance
# =====================================================

print("Filas antes de limpieza profunda:", remittance.shape[0])

# 1) Eliminar la fila donde aparece "VOUCHER  Tipo de Documento ..."
remittance = remittance[~(
    remittance["Tipo de Documento"].astype(str).str.upper().str.contains("DESCRIPCION")
)]

# 2) Eliminar filas donde la primera columna es "VOUCHER" seguido de header
remittance = remittance[~(
    remittance[remittance.columns[0]].astype(str).str.upper().str.contains("VOUCHER") &
    remittance["Importe_clean"].isna()
)]

print("Después de quitar headers residuales:", remittance.shape[0])

# 3) Quitar filas con importe = 0 (si no corresponden a pagos)
remittance = remittance[remittance["Importe_clean"] != 0]

print("Después de quitar importes = 0:", remittance.shape[0])

# 4) Quitar columnas cuyo nombre sea solo un número perdido (como el '3')
cols = [c for c in remittance.columns if not str(c).isdigit()]
remittance = remittance[cols]

# 5) Reset final de índice
remittance = remittance.reset_index(drop=True)

print("\n=== Remittance FINAL ===")
print(remittance.head(20))
print("\nTotal filas finales:", remittance.shape)


Filas antes de limpieza profunda: 677
Después de quitar headers residuales: 677
Después de quitar importes = 0: 12

=== Remittance FINAL ===
3  VOUCHER    Tipo de Documento                  DOCUMENTOTIENDA SECCION  \
0       FS        FACTURA VENTA  VPP1 2004529ADM. FIDELIDAD - SE           
1       FS        FACTURA VENTA  VPP1 2004621ADM. FIDELIDAD - SE           
2       FS        FACTURA VENTA  VPP1 2004622ADM. FIDELIDAD - SE           
3       FS        FACTURA VENTA    VPP2 2021716ADM. JUMBO - SEDE           
4       FS        FACTURA VENTA    VPP2 2022342ADM. JUMBO - SEDE           
5       CH  Costo de Transferen                ADM. JUMBO - SEDE           
6       FS        FACTURA VENTA  VPP1 2004529ADM. FIDELIDAD - SE           
7       FS        FACTURA VENTA  VPP1 2004621ADM. FIDELIDAD - SE           
8       FS        FACTURA VENTA  VPP1 2004622ADM. FIDELIDAD - SE           
9       FS        FACTURA VENTA    VPP2 2021716ADM. JUMBO - SEDE           
10      FS        FACTU

In [11]:
remittance.head(20)

3,VOUCHER,Tipo de Documento,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,Importe de Remittance,DOC.SOPORTE,Importe_clean
0,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0,5.723900e+07
1,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0,6.426000e+06
2,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0,4.165000e+07
3,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0,2.507098e+09
4,FS,FACTURA VENTA,VPP2 2022342ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0,1.075965e+09
5,CH,Costo de Transferen,ADM. JUMBO - SEDE,,12/11/2025,3.000,0,0,0,0,0,3.000,0,3.000000e+03
6,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0,5.723900e+07
7,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0,6.426000e+06
8,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0,4.165000e+07
9,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0,2.507098e+09


In [12]:
    remittance = remittance.rename(columns={
        "DESCRIPCION": "Tipo de Documento",
        "DOCUMENTO": "Referencia / Factura",
        "VALOR PAG": "Importe de Remittance"
    })

In [15]:
columnas_finales = [
    "VOUCHER",
    "Tipo de Documento",
    "Referencia / Factura",
    "SECCION",
    "Importe de Remittance",
    "DOC.SOPORTE"
]
remittance = remittance[columnas_finales]


KeyError: "['Referencia / Factura'] not in index"

In [13]:
remittance.head()

3,VOUCHER,Tipo de Documento,DOCUMENTOTIENDA,SECCION,F. REGISTRO,VALOR FAC.,IVA FAC.,RET. FUENTE,RET. IVA,RET. ICA,OTROS IMP.,Importe de Remittance,DOC.SOPORTE,Importe_clean
0,FS,FACTURA VENTA,VPP1 2004529ADM. FIDELIDAD - SE,,04/09/2025,48.364.550,8.874.450,0,0,0,5.291.000,57.239.000,0,5.723900e+07
1,FS,FACTURA VENTA,VPP1 2004621ADM. FIDELIDAD - SE,,17/09/2025,5.429.700,996.300,0,0,0,594.000,6.426.000,0,6.426000e+06
2,FS,FACTURA VENTA,VPP1 2004622ADM. FIDELIDAD - SE,,17/09/2025,35.192.500,6.457.500,0,0,0,3.850.000,41.650.000,0,4.165000e+07
3,FS,FACTURA VENTA,VPP2 2021716ADM. JUMBO - SEDE,,01/11/2025,2.118.392.716,388.705.576,0,0,0,84.272.211,2.507.098.292,0,2.507098e+09
4,FS,FACTURA VENTA,VPP2 2022342ADM. JUMBO - SEDE,,09/11/2025,909.145.377,166.819.814,0,0,0,36.166.897,1.075.965.191,0,1.075965e+09


In [23]:
# =====================================================
# 5. Filtrar VOUCHERS válidos
# =====================================================

filter_values = [
    "DAT","CH","DAV","DCA","DCC","DCF","DEV","DND","DPC",
    "FPM","FS","LTG","RPL","VOUCHER"
]

remittance = df_all[df_all[df_all.columns[0]].isin(filter_values)].copy()
remittance = remittance.reset_index(drop=True)

print("Filtrado por VOUCHERS — filas:", remittance.shape[0])
remittance.head()


SyntaxError: unexpected EOF while parsing (522034261.py, line 15)

In [10]:
# Lectura del PDF Remittance (solo para validar)
reader = PdfReader(archivo_remittance)
print(f"\nPáginas del Remittance PDF: {len(reader.pages)}")

# Lectura del FBL5N
df_fbl5n_raw = pd.read_excel(archivo_fbl5n)
print("FBL5N cargado:", df_fbl5n_raw.shape)


Páginas del Remittance PDF: 39
FBL5N cargado: (41, 24)


In [14]:
# ==========================================================
# 2. EXTRAER TABLAS DEL REMITTANCE COMO EN TU SCRIPT ORIGINAL
# ==========================================================

print("Extrayendo tablas con Camelot...")

tables_stream = camelot.read_pdf(
    archivo_remittance, 
    pages="all", 
    flavor="stream",
    strip_text="\n"
)

stream_pages = set(int(t.page) for t in tables_stream)
all_pages = set(range(1, len(reader.pages)+1))
missing_pages = all_pages - stream_pages

# Extraer con lattice si faltan páginas
tables_lattice = []
if missing_pages:
    tables_lattice = camelot.read_pdf(
        archivo_remittance,
        pages=",".join(str(p) for p in missing_pages),
        flavor="lattice",
        strip_text="\n"
    )

# Unir todas las tablas
df_all = pd.concat(
    [t.df for t in list(tables_stream) + list(tables_lattice)],
    ignore_index=True
)

print("Tablas unidas:", df_all.shape)


# ==========================================================
# 3. DETECTAR HEADER REAL
# ==========================================================

header_idx_candidates = df_all.apply(
    lambda row: row.astype(str).str.contains("DOCUMENTO", case=False).any()
             or row.astype(str).str.contains("VOUCHER", case=False).any(),
    axis=1
)

if not header_idx_candidates.any():
    print("❌ ERROR: No se detectó encabezado")
else:
    header_row_idx = header_idx_candidates.idxmax()
    print("Header detectado en fila:", header_row_idx)

# Aplicar header
df_all.columns = df_all.iloc[header_row_idx]
df_all = df_all.drop(index=list(range(0, header_row_idx + 1))).reset_index(drop=True)

# Quitar columnas nulas
df_all = df_all.loc[:, ~df_all.columns.isna()]


# ==========================================================
# NORMALIZAR COLUMNAS (versión 100% compatible)
# ==========================================================

cols = pd.Series(df_all.columns.astype(str))
cols = cols.str.replace(r"[\n\r]+", " ", regex=True)
cols = cols.str.replace(r"\s+", " ", regex=True)
cols = cols.str.strip()
cols = cols.str.upper()
df_all.columns = cols

print("Columnas normalizadas:")
print(df_all.columns.tolist())


# ==========================================================
# ⚠️ ELIMINAR HEADERS REPETIDOS DENTRO DEL PDF
# ==========================================================

header_keywords = [
    "DESCRIPCION", "DOCUMENTO", "VALOR", "PAG", "FACTURA",
    "VOUCHER", "SECCION", "DOC", "SOPORTE"
]

def is_header_row(row):
    first = str(row.iloc[0]).upper().strip()
    return any(k in first for k in header_keywords)

df_all = df_all[~df_all.apply(is_header_row, axis=1)].reset_index(drop=True)


# ==========================================================
# 4. FILTRAR SOLO LOS VOUCHERS VÁLIDOS
# ==========================================================

filter_values = ['DAT','CH','DAV','DCA','DCC','DCF','DEV','DND',
                 'DPC','FPM','FS','LTG','RPL','VOUCHER']

remittance = df_all[df_all[df_all.columns[0]].isin(filter_values)].copy()

print("Filas válidas remittance:", remittance.shape)


# ==========================================================
# 5. RENOMBRAR COLUMNAS CRÍTICAS
# ==========================================================

renombres = {
    "DESCRIPCION": "Tipo de Documento",
    "DOCUMENTO": "Referencia / Factura",
    "VALOR PAG": "Importe de Remittance"
}

remittance = remittance.rename(columns={c: renombres[c] for c in renombres if c in remittance.columns})

print("\nColumnas actuales luego de renombrar:")
print(remittance.columns.tolist())


# ==========================================================
# 6. LIMPIEZA DEL IMPORTE (100% NUMÉRICO)
# ==========================================================

def clean_importe(valor):
    if pd.isna(valor):
        return np.nan
    
    s = str(valor).strip()

    # Quitar textos contables
    s = s.replace(" ", "").replace("(CR)", "-").replace("(DR)", "")

    # Si el negativo está al final → mover al inicio
    if s.endswith("-"):
        s = "-" + s[:-1]

    # Quitar cualquier cosa que no sea número, coma o menos
    s = re.sub(r"[^0-9,-]", "", s)

    # Quitar puntos de miles y convertir coma en punto
    s = s.replace(".", "")
    s = s.replace(",", ".")

    try:
        return float(s)
    except:
        return np.nan

remittance["Importe_clean"] = remittance["Importe de Remittance"].apply(clean_importe)

print("\nMuestra de importes:")
print(remittance[["Importe de Remittance", "Importe_clean"]].head(20))

print("\nCantidad de importes nulos después de limpiar:")
print(remittance["Importe_clean"].isna().sum())

# ==========================================================
# 7. FILTRADO FINAL DEL REMITTANCE
# ==========================================================

# 1. Eliminar filas donde el importe sea NaN
remittance = remittance[~remittance["Importe_clean"].isna()].copy()

# 2. Eliminar importes 0 (son basura del PDF)
remittance = remittance[remittance["Importe_clean"] != 0].copy()

# 3. Eliminar filas donde la factura/referencia esté vacía
if "Referencia / Factura" in remittance.columns:
    remittance = remittance[remittance["Referencia / Factura"].notna()]
    remittance = remittance[remittance["Referencia / Factura"].astype(str).str.strip() != ""]
else:
    print("⚠️ No se encontró la columna de Factura")

# 4. Eliminar duplicados completos
remittance = remittance.drop_duplicates()

# 5. Reset de índices
remittance = remittance.reset_index(drop=True)

print("\n==========================================================")
print(" REMITTANCE FINAL LIMPIO ")
print("==========================================================")
print(remittance.head(20))
print("\nTotal de registros finales:", len(remittance))


Extrayendo tablas con Camelot...
Tablas unidas: (2293, 14)
Header detectado en fila: 3
Columnas normalizadas:
['VOUCHER', 'DESCRIPCION', 'DOCUMENTOTIENDA', 'SECCION', 'F. REGISTRO', 'VALOR FAC.', 'IVA FAC.', 'RET. FUENTE', 'RET. IVA', 'RET. ICA', 'OTROS IMP.', 'VALOR PAG', 'DOC.SOPORTE']
Filas válidas remittance: (677, 13)

Columnas actuales luego de renombrar:
['VOUCHER', 'Tipo de Documento', 'DOCUMENTOTIENDA', 'SECCION', 'F. REGISTRO', 'VALOR FAC.', 'IVA FAC.', 'RET. FUENTE', 'RET. IVA', 'RET. ICA', 'OTROS IMP.', 'Importe de Remittance', 'DOC.SOPORTE']

Muestra de importes:
3   Importe de Remittance  Importe_clean
0              57.239.000   5.723900e+07
1               6.426.000   6.426000e+06
2              41.650.000   4.165000e+07
3           2.507.098.292   2.507098e+09
4           1.075.965.191   1.075965e+09
5                   3.000   3.000000e+03
31             57.239.000   5.723900e+07
32              6.426.000   6.426000e+06
33             41.650.000   4.165000e+07
34     